https://github.com/destination-earth-digital-twins/polytope-examples/blob/main/climate-dt/explorer/04_lazy_browse_portfolio_hourly.ipynb

https://github.com/destination-earth-digital-twins/polytope-examples/blob/main/climate-dt/explorer/04_lazy_browse_portfolio_hourly.ipynb

In [2]:
%%capture cap
%run ../src/desp-authentication.py -u "" -p ""

In [3]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

In [4]:
S3_KEY = ""
S3_SECRET = ""

In [5]:
import earthkit.data
import earthkit.plots
import earthkit.geo.cartography

import logging, warnings
import earthkit.data

# Disable earthkit disk cache (polytope_zarr caches decoded arrays in memory)
earthkit.data.config.set("cache-policy", "off")

# Silence verbose output from polytope / earthkit internals
for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from polytope_zarr import PolytopeZarrStore

import s3fs
import io



In [6]:
# Defaults to making a live data request. Set to false to use the cached GRIB file instead.
import os

LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

True

In [8]:
eodc_s3 = s3fs.S3FileSystem(
    key=S3_KEY,
    secret=S3_SECRET,
    client_kwargs={
        "endpoint_url": "https://objects.eodc.eu"
    })

path = "destine-climate-dt/2t/netcdf"

### Realization 1

In [9]:
realization="1"

story_store1 = PolytopeZarrStore.from_climate_dt(
    models=["IFS-FESOM"],
    experiment=["cont", "hist", "Tplus2.0K"],
    activity="story-nudging",
    levtype="sfc",
    frequency="hourly",
    start_date="2017-01-01T00:00:00",
    end_date="2026-07-31T23:00:00",
    resolution="high",
    realization=realization,
)
print(story_store1)
story_store1._filter_hours = [12]
# story_store1._batch_dim = None


<PolytopeZarrStore 34 variables (time=83976, cell=3145728, climate=3)>


In [10]:
ds_story1 = story_store1.open()
ds_story1

<xarray.Dataset> Size: 108TB
Dimensions:       (climate: 3, time: 83976, cell: 3145728)
Coordinates:
  * cell          (cell) int32 13MB 0 1 2 3 ... 3145724 3145725 3145726 3145727
  * climate       (climate) object 24B 'cont' 'hist' 'Tplus2.0K'
  * time          (time) datetime64[ns] 672kB 2017-01-01 ... 2026-07-31T23:00:00
Data variables: (12/34)
    10si          (climate, time, cell) float32 3TB ...
    10u           (climate, time, cell) float32 3TB ...
    10v           (climate, time, cell) float32 3TB ...
    2d            (climate, time, cell) float32 3TB ...
    2t            (climate, time, cell) float32 3TB ...
    avg_ie        (climate, time, cell) float32 3TB ...
    ...            ...
    sp            (climate, time, cell) float32 3TB ...
    tcc           (climate, time, cell) float32 3TB ...
    tciw          (climate, time, cell) float32 3TB ...
    tclw          (climate, time, cell) float32 3TB ...
    tcw           (climate, time, cell) float32 3TB ...
    tcwv          (climate, time, cell) float32 3TB ...
Attributes:
    _polytope_store:  <PolytopeZarrStore 34 variables (time=83976, cell=31457...

In [11]:
import earthkit.geo.cartography

COUNTRY = "Austria"
shapes = earthkit.geo.cartography.country_polygons([COUNTRY], resolution=50e6)

In [18]:
climate = "Tplus2.0K"

vsw_cont = ds_story1["2t"].polytope.sel(
        climate=climate, time=slice("2017-01-01", "2026-07-31"), polygon=shapes,
    )
data_bytes = vsw_cont.to_netcdf()  # no path → returns bytes
with eodc_s3.open(f"{path}/vsw_story-nudging_{climate}_r{realization}.nc", "wb") as f:
    f.write(data_bytes)
    print(f"{path}/vsw_story-nudging_{climate}_r{realization}.nc")

  🌍 polygon request for 2t (20170101/to/20260731)


destine-climate-dt/2t/netcdf/vsw_story-nudging_Tplus2.0K_r1.nc


In [19]:
vsw_cont

<xarray.Dataset> Size: 15MB
Dimensions:    (time: 3499, points: 517)
Coordinates:
  * time       (time) datetime64[ns] 28kB 2017-01-01T12:00:00 ... 2026-07-31T...
  * points     (points) int64 4kB 0 1 2 3 4 5 6 ... 510 511 512 513 514 515 516
    latitude   (points) float64 4kB 46.47 46.47 46.47 ... 48.92 48.92 48.92
    longitude  (points) float64 4kB 14.23 14.42 14.61 ... 15.03 15.24 15.44
    levelist   (points) float64 4kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Data variables:
    2t         (time, points) float64 14MB 278.0 278.1 278.8 ... 307.6 308.3
Attributes: (12/15)
    activity:     story-nudging
    class:        d1
    dataset:      climate-dt
    experiment:   tplus2.0k
    expver:       0001
    generation:   2
    ...           ...
    resolution:   high
    stream:       clte
    type:         fc
    number:       0
    step:         0
    date:         2017-01-01 12:00:00Z

In [ ]:
climate = "cont"

vsw_cont_by_level = {}
for level in [1, 2, 3]:
    vsw_cont = ds_story1["vsw"].polytope.sel(
        climate=climate, time=slice("2017-01-01", "2025-12-31"),
        level=level, polygon=shapes,
    )
    data_bytes = vsw_cont.to_netcdf()  # no path → returns bytes
    with eodc_s3.open(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc", "wb") as f:
        f.write(data_bytes)
        print(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc")

  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_cont_level1_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_cont_level2_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_cont_level3_r1.nc


In [44]:
climate = "hist"

vsw_cont_by_level = {}
for level in [1, 2, 3]:
    vsw_cont = ds_story1["vsw"].polytope.sel(
        climate=climate, time=slice("2017-01-01", "2025-12-31"),
        level=level, polygon=shapes,
    )
    data_bytes = vsw_cont.to_netcdf()  # no path → returns bytes
    with eodc_s3.open(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc", "wb") as f:
        f.write(data_bytes)
        print(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc")

  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_hist_level1_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_hist_level2_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_hist_level3_r1.nc


In [45]:
climate = "Tplus2.0K"

vsw_cont_by_level = {}
for level in [1, 2, 3]:
    vsw_cont = ds_story1["vsw"].polytope.sel(
        climate=climate, time=slice("2017-01-01", "2025-12-31"),
        level=level, polygon=shapes,
    )
    data_bytes = vsw_cont.to_netcdf()  # no path → returns bytes
    with eodc_s3.open(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc", "wb") as f:
        f.write(data_bytes)
        print(f"{path}/vsw_story-nudging_{climate}_level{level}_r{realization}.nc")

  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_Tplus2.0K_level1_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_Tplus2.0K_level2_r1.nc
  🌍 polygon request for vsw (20170101/to/20170131)


destine-climate-dt/vsw/netcdf/vsw_story-nudging_Tplus2.0K_level3_r1.nc


In [12]:
import earthkit.geo.cartography

COUNTRY = "Austria"
shapes = earthkit.geo.cartography.country_polygons([COUNTRY], resolution=50e6)

In [10]:
climate = "hist"
path = "destine-climate-dt/sd/netcdf"

vsw_cont = ds_story1["sd"].polytope.sel(
        climate=climate, time=slice("2017-01-01", "2026-07-31"),
    )

# vsw_cont_by_level = {}
# for level in [1, 2, 3]:
    
# data_bytes = vsw_cont.to_netcdf()  # no path → returns bytes
# with eodc_s3.open(f"{path}/vsw_story-nudging_{climate}_r{realization}.nc", "wb") as f:
#     f.write(data_bytes)
#     print(f"{path}/vsw_story-nudging_{climate}_r{realization}.nc")

In [11]:
vsw_cont

<xarray.DataArray 'sd' (time: 83976, cell: 3145728)> Size: 1TB
[264165654528 values with dtype=float32]
Coordinates:
  * cell     (cell) int32 13MB 0 1 2 3 4 ... 3145724 3145725 3145726 3145727
    climate  <U4 16B 'hist'
  * time     (time) datetime64[ns] 672kB 2017-01-01 ... 2026-07-31T23:00:00
Attributes:
    long_name:        Snow depth water equivalent
    units:            kg m**-2
    _polytope_store:  <PolytopeZarrStore 34 variables (time=83976, cell=31457...

In [12]:
import numpy as np
from astropy_healpix import HEALPix
import astropy.units as u
from shapely.geometry import Polygon, MultiPolygon
from shapely.vectorized import contains

hp = HEALPix(nside=story_store1.nside, order="nested")
n_cells = vsw_cont.sizes["cell"]
lon, lat = hp.healpix_to_lonlat(np.arange(n_cells))
lon_deg = ((lon.to_value(u.deg) + 180) % 360) - 180
lat_deg = lat.to_value(u.deg)

polygons = [Polygon([(lo, la) for la, lo in ring]) for ring in shapes]
country_shape = polygons[0] if len(polygons) == 1 else MultiPolygon(polygons)
mask = contains(country_shape, lon_deg, lat_deg)
cell_idx = np.nonzero(mask)[0]

vsw_cont_clipped = vsw_cont.isel(cell=cell_idx)
vsw_cont_clipped.attrs.pop("_polytope_store", None)  # not netCDF-serializable
vsw_cont_clipped

<xarray.DataArray 'sd' (time: 83976, cell: 517)> Size: 174MB
[43415592 values with dtype=float32]
Coordinates:
  * cell     (cell) int32 2kB 171863 171867 171868 ... 183078 183080 183081
    climate  <U4 16B 'hist'
  * time     (time) datetime64[ns] 672kB 2017-01-01 ... 2026-07-31T23:00:00
Attributes:
    long_name:  Snow depth water equivalent
    units:      kg m**-2

In [13]:
vsw_cont_clipped = vsw_cont_clipped.sel(time=vsw_cont_clipped["time"].dt.hour == 12)
vsw_cont_clipped

<xarray.DataArray 'sd' (time: 3499, cell: 517)> Size: 7MB
[1808983 values with dtype=float32]
Coordinates:
  * cell     (cell) int32 2kB 171863 171867 171868 ... 183078 183080 183081
    climate  <U4 16B 'hist'
  * time     (time) datetime64[ns] 28kB 2017-01-01T12:00:00 ... 2026-07-31T12...
Attributes:
    long_name:  Snow depth water equivalent
    units:      kg m**-2

In [14]:
story_store1._batch_dim = None

data_bytes = vsw_cont_clipped.to_netcdf()  # no path → returns bytes


2026-09-24 14:21:16 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-09-24 14:21:16 - INFO - Sending request...
{'request': 'activity: story-nudging\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            "date: '20170101'\n"
            'experiment: hist\n'
            "expver: '0001'\n"
            "generation: '2'\n"
            'levtype: sfc\n'
            'model: IFS-FESOM\n'
            'param: sd\n'
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            "time: '1200'\n"
            'type: fc\n',
 'verb': 'retrieve'}
2026-09-24 14:21:17 - INFO - Polytope user key found in session cache for user koenifra
2026-09-24 14:21:17 - INFO - Request accepted. Please poll ./01f3z8k9838283t008ns4qeapx for status
2026-09-24 14:21:17 - INFO - Polytope user key found in session cache for user koenifra
2026-09-24 14:21:17 - INFO - Checking request status (01f3z8k9838283t008ns4qeapx)...
2026-09-24 14:21:18 - INFO - 

: 

In [ ]:
# with eodc_s3.open(f"{path}/sd_story-nudging_{climate}_r{realization}.nc", "wb") as f:
#     f.write(data_bytes)
#     print(f"{path}/sd_story-nudging_{climate}_r{realization}.nc")